
# PRE-01 v3.4 — UNSW-NB15 Source + Fingerprint-Multiplicity Stratified Partitioning

## Singleton-tolerance implementation fix — frozen v3 scientific policy unchanged

v3.3 successfully found a coordinated non-singleton assignment, but the next cell incorrectly required the **non-singleton total** in every role to equal the exact target implied by the singleton band.

That check was stricter than the frozen amendment.

The frozen policy permits the singleton band an absolute deviation of **±1 row** because `M_1 = 1`. The coordinated v3.3 solver correctly returned role residuals equivalent to singleton-band deviations:

- `benign_weight_train`: 0
- `benign_early_stop`: +1 singleton row
- `benign_threshold_calibration`: −1 singleton row
- `development`: +1 singleton row
- `final_test`: −1 singleton row

All are **inside the frozen ±1 singleton tolerance**.

v3.4 therefore changes only the post-solver validation and singleton repair logic:

1. compute the actual singleton rows required by each role from its exact role capacity minus its solved non-singleton total;
2. verify that this actual singleton count differs from the frozen singleton-band target by at most 1 row;
3. allocate singleton fingerprints to those actual residual capacities;
4. continue to require exact final role capacities, exact overall source totals, and the existing singleton source×band ±1 gates.

No scientific constraint is relaxed. This fix makes the implementation conform to the already-frozen policy instead of imposing an accidental exact-equality rule.

**Formal Notebook 01 remains blocked.**


In [1]:

# ============================================================
# V3.4-0. IMPORTS / CONSTANTS / FROZEN POLICY IDENTITY
# ============================================================
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, sys, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy
from scipy.optimize import milp, Bounds, LinearConstraint, linprog
from scipy.sparse import coo_matrix

DATASET_NAME="UNSW-NB15"
PROTOCOL_VERSION="Experiment_1_v2"
PARTITION_SEED=20260914
MODEL_BRANCH_VERSION="v3_source_multiplicity_20260919"

PROJECT_FOLDER_NAME="EXPERIMENT1_V2"
PRIMARY_PROJECT=Path("/content/drive/MyDrive")/PROJECT_FOLDER_NAME
FALLBACK_PROJECT=Path("/content/exp1v2_google_drive/MyDrive")/PROJECT_FOLDER_NAME
PROJECT_ROOT_OVERRIDE=None

PARENT_PRE01_DIRNAME="partitioning_s8_s11_v2_source_stratified_seed20260914"
V3_PRE01_DIRNAME="partitioning_s8_s11_v3_source_multiplicity_seed20260914"
AUDIT_00E_STAGE="preformal_fingerprint_multiplicity_feasibility_audit"
AUDIT_BRANCH="v2_source_stratified_20260918"

EXPECTED_PARENT_MEMBERSHIP_SHA="3114e887aeff514be147d68a9f938d9c889d2c89a21a1ccffd6ff65fcb4a4fac"
EXPECTED_00D_MANIFEST_HASH="2671c9e03f72690f08833bea010c756da3b92a00b1afd9943e1044c35f41c2d5"
EXPECTED_00E_MANIFEST_HASH="d145b95f1eea0c81ae5465ba3d7e9ddabda0b9b713c82df8d4f8e13541f46449"
EXPECTED_ON_DRIVE_V3_POLICY_HASH="17122c41630be246c40d9018a97d6f80b58d7422f438673594acb74fd4d2d3fb"

ROLE_ORDER=["benign_weight_train","benign_early_stop","benign_threshold_calibration","development","final_test"]
BAND_ORDER=["1","2","3-4","5-9","10-19","20-49","50-99",">=100"]
M_B={"1":1,"2":2,"3-4":4,"5-9":9,"10-19":19,"20-49":49,"50-99":99,">=100":760}
EXEMPT_SOURCE_BAND={("UNSW-NB15_1.csv",">=100"),("UNSW-NB15_2.csv",">=100")}
PURE_BENIGN_CLASS="pure benign group"

def require(cond,msg):
    if not bool(cond):
        print("\nPRE-01 v3.4 INTEGRITY: FAIL")
        print("No v3 membership is authorized for downstream use.")
        raise RuntimeError(str(msg))

def sha256_file(path, chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def canonical_hash(obj):
    return hashlib.sha256(
        json.dumps(obj,sort_keys=True,separators=(",",":"),ensure_ascii=False).encode("utf-8")
    ).hexdigest()

def atomic_json(path,obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    with open(tmp,"w",encoding="utf-8") as f: json.dump(obj,f,indent=2,ensure_ascii=False)
    os.replace(tmp,path)

def atomic_csv(path,df):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp"); df.to_csv(tmp,index=False); os.replace(tmp,path)

def size_band(n):
    n=int(n)
    if n==1:return "1"
    if n==2:return "2"
    if n<=4:return "3-4"
    if n<=9:return "5-9"
    if n<=19:return "10-19"
    if n<=49:return "20-49"
    if n<=99:return "50-99"
    return ">=100"

def tie_key(fp):
    return int.from_bytes(hashlib.sha256(f"{PARTITION_SEED}|{fp}".encode()).digest()[:8],"big")

require(hasattr(scipy.optimize,"milp"),f"SciPy {scipy.__version__} lacks scipy.optimize.milp")
print("SciPy:",scipy.__version__)
print("Formal Notebook 01 remains BLOCKED.")


SciPy: 1.16.3
Formal Notebook 01 remains BLOCKED.


In [2]:

# ============================================================
# V3.4-1. MOUNT / VERIFY PARENT / VERIFY ALREADY-FROZEN V3 POLICY
# ============================================================
def resolve_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        p=Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        require(p.exists(),f"PROJECT_ROOT_OVERRIDE does not exist: {p}")
        return p
    if PRIMARY_PROJECT.exists(): return PRIMARY_PROJECT.resolve()
    try:
        from google.colab import drive
        drive.mount("/content/drive",force_remount=False)
    except Exception as exc:
        print("Drive mount attempt:",repr(exc))
    if PRIMARY_PROJECT.exists(): return PRIMARY_PROJECT.resolve()
    if FALLBACK_PROJECT.exists(): return FALLBACK_PROJECT.resolve()
    try:
        from google.colab import drive
        drive.mount("/content/exp1v2_google_drive",force_remount=False)
    except Exception as exc:
        print("Fallback mount attempt:",repr(exc))
    require(FALLBACK_PROJECT.exists(),"Could not locate EXPERIMENT1_V2.")
    return FALLBACK_PROJECT.resolve()

PROJECT_ROOT=resolve_project_root()
DATASET_ROOT=PROJECT_ROOT/"DATASETS"/DATASET_NAME
PARENT_ROOT=DATASET_ROOT/PARENT_PRE01_DIRNAME
PARENT_MEMBERSHIP=PARENT_ROOT/"partition_membership.parquet"
V3_ROOT=DATASET_ROOT/V3_PRE01_DIRNAME
V3_ROOT.mkdir(parents=True,exist_ok=True)

AUDIT_00E_ROOT=DATASET_ROOT/"modelling"/AUDIT_00E_STAGE/AUDIT_BRANCH
AUDIT_00E_MANIFEST=AUDIT_00E_ROOT/"diagnostic_manifest.json"
AUDIT_GLOBAL_CELL=AUDIT_00E_ROOT/"global_source_x_multiplicity_cells.csv"

require(PARENT_MEMBERSHIP.exists(),"Missing parent corrected-v2 membership.")
require(sha256_file(PARENT_MEMBERSHIP)==EXPECTED_PARENT_MEMBERSHIP_SHA,
        "Parent corrected-v2 membership SHA mismatch.")
require(AUDIT_00E_MANIFEST.exists() and AUDIT_GLOBAL_CELL.exists(),"Missing 00E evidence.")

with open(AUDIT_00E_MANIFEST,"r",encoding="utf-8") as f: audit=json.load(f)
require(audit.get("diagnostic_manifest_hash")==EXPECTED_00E_MANIFEST_HASH,"00E manifest mismatch.")
require(audit.get("partition_membership_sha256")==EXPECTED_PARENT_MEMBERSHIP_SHA,
        "00E was run on a different parent membership.")

# The first v3 attempt wrote the amendment before failing, exactly as intended.
policy_path=V3_ROOT/"source_multiplicity_amendment.json"
require(policy_path.exists(),
        "Expected frozen v3 amendment is absent. Run the original v3 notebook only through the policy-freeze cell or restore the amendment artifact.")
with open(policy_path,"r",encoding="utf-8") as f: FROZEN_POLICY=json.load(f)
require(FROZEN_POLICY.get("policy_hash")==EXPECTED_ON_DRIVE_V3_POLICY_HASH,
        f"On-Drive frozen v3 policy hash mismatch: {FROZEN_POLICY.get('policy_hash')}")
require(FROZEN_POLICY.get("parent_membership_sha256")==EXPECTED_PARENT_MEMBERSHIP_SHA,
        "Frozen v3 policy points to a different parent membership.")
require(FROZEN_POLICY.get("evidence_00D_manifest_hash")==EXPECTED_00D_MANIFEST_HASH,
        "Frozen v3 policy 00D evidence mismatch.")
require(FROZEN_POLICY.get("evidence_00E_manifest_hash")==EXPECTED_00E_MANIFEST_HASH,
        "Frozen v3 policy 00E evidence mismatch.")

require(not (V3_ROOT/"partition_membership.parquet").exists(),
        "A v3 membership already exists. Refusing to create another membership in the same scientific branch.")

implementation_note={
    "dataset":DATASET_NAME,
    "amendment_id":FROZEN_POLICY["amendment_id"],
    "scientific_policy_hash":FROZEN_POLICY["policy_hash"],
    "implementation_version":"v3.4_singleton_tolerance_fix",
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "previous_v3_failure":"greedy repair residual [-3621, 926, 883, 893, 919]",
    "previous_v3_1_failure":"monolithic MILP 900-second time limit; no primal solution",
    "previous_v3_2_failure":"low-band bridge infeasible after independently solved high bands",
    "previous_v3_3_failure":"post-solver check incorrectly required exact non-single totals instead of allowing the frozen singleton-band ±1 tolerance",
    "membership_created_by_previous_attempts":False,
    "scientific_policy_changed":False,
    "change":"Use actual singleton residual capacities and validate them against the frozen M_1=1 tolerance."
}
implementation_note["implementation_note_hash"]=canonical_hash(
    {k:v for k,v in implementation_note.items() if k not in {"created_at_utc","implementation_note_hash"}}
)
atomic_json(V3_ROOT/"implementation_fix_v3_4.json",implementation_note)

print("PASS: frozen scientific amendment preserved unchanged.")
print("Scientific policy hash:",FROZEN_POLICY["policy_hash"])
print("Implementation note:",implementation_note["implementation_note_hash"])


Mounted at /content/drive
PASS: frozen scientific amendment preserved unchanged.
Scientific policy hash: 17122c41630be246c40d9018a97d6f80b58d7422f438673594acb74fd4d2d3fb
Implementation note: 83d1c4b163587747a9460da72b4b9c6381dda83dc52f9ade8e179c045e764206


In [3]:

# ============================================================
# V3.4-2. LOAD STRUCTURE / GROUPS / SOURCE VECTORS
# ============================================================
schema=pq.read_schema(PARENT_MEMBERSHIP)
needed={"row_id","feature_fingerprint","fingerprint_group_class","partition","source_file"}
require(needed.issubset(set(schema.names)),f"Parent membership missing {sorted(needed-set(schema.names))}")

struct=pd.read_parquet(PARENT_MEMBERSHIP,
    columns=["row_id","feature_fingerprint","fingerprint_group_class","partition","source_file"])
require(not struct["row_id"].duplicated().any(),"Duplicate row_id.")

pure=struct.loc[struct["fingerprint_group_class"].astype(str).eq(PURE_BENIGN_CLASS)].copy()
require(len(pure)==2217778,f"Unexpected pure-benign row count: {len(pure)}")

group=(pure.groupby("feature_fingerprint",sort=False)
          .agg(group_size=("row_id","size"),old_partition=("partition","first"))
          .reset_index())
require(int(pure.groupby("feature_fingerprint")["partition"].nunique().max())==1,
        "Parent pure-benign group crosses roles.")
group["size_band"]=group["group_size"].map(size_band)
group["tie_key"]=[tie_key(str(x)) for x in group["feature_fingerprint"]]

sources=sorted(pure["source_file"].astype(str).unique().tolist())
require(sources==["UNSW-NB15_1.csv","UNSW-NB15_2.csv","UNSW-NB15_3.csv","UNSW-NB15_4.csv"],
        f"Unexpected source set: {sources}")

src_counts=(pure.groupby(["feature_fingerprint","source_file"],sort=False)
                 .size().rename("rows_in_source").reset_index())
src_wide=(src_counts.pivot(index="feature_fingerprint",columns="source_file",values="rows_in_source")
                    .fillna(0).reindex(columns=sources,fill_value=0).astype(int))
group=group.set_index("feature_fingerprint").join(src_wide).reset_index()

role_caps=(pure.groupby("partition").size().reindex(ROLE_ORDER).astype(int))
require(int(role_caps.sum())==len(pure),"Role capacities do not reconcile.")

singleton=group.loc[group["group_size"].eq(1)].copy()
non_single=group.loc[group["group_size"].gt(1)].copy()
singleton_supply=singleton.groupby("source_file" if "source_file" in singleton.columns else "feature_fingerprint").size()

print("Pure-benign rows:",f"{len(pure):,}")
print("Pure-benign groups:",f"{len(group):,}")
print("Non-singleton groups:",f"{len(non_single):,}")
print("Singleton groups:",f"{len(singleton):,}")


Pure-benign rows: 2,217,778
Pure-benign groups: 1,938,413
Non-singleton groups: 56,168
Singleton groups: 1,882,245


In [4]:

# ============================================================
# V3.4-3. TARGET MATRICES + SINGLETON SOURCE IDENTITIES
# ============================================================
def rounded_transport_targets(row_totals,col_totals,row_labels,col_labels):
    row=np.asarray(row_totals,dtype=int); col=np.asarray(col_totals,dtype=int)
    require(int(row.sum())==int(col.sum()),"Transport margins differ.")
    N=int(row.sum()); expected=np.outer(row,col)/N; base=np.floor(expected).astype(int)
    rdef=row-base.sum(axis=1); cdef=col-base.sum(axis=0); frac=expected-base
    nR,nC=base.shape
    rows=[]; cols=[]; data=[]; beq=[]
    rr=0
    for i in range(nR):
        for j in range(nC):
            rows.append(rr); cols.append(i*nC+j); data.append(1.0)
        beq.append(float(rdef[i])); rr+=1
    for j in range(nC):
        for i in range(nR):
            rows.append(rr); cols.append(i*nC+j); data.append(1.0)
        beq.append(float(cdef[j])); rr+=1
    A=coo_matrix((data,(rows,cols)),shape=(rr,nR*nC)).tocsr()
    res=linprog(-frac.ravel(),A_eq=A,b_eq=np.asarray(beq),bounds=[(0,1)]*(nR*nC),method="highs")
    require(res.success,f"Target rounding failed: {res.message}")
    add=np.rint(res.x).astype(int).reshape(nR,nC)
    require(np.max(np.abs(res.x-add.ravel()))<1e-7,"Transport rounding not integral.")
    out=base+add
    require(np.array_equal(out.sum(axis=1),row),"Target row margins failed.")
    require(np.array_equal(out.sum(axis=0),col),"Target col margins failed.")
    return pd.DataFrame(out,index=row_labels,columns=col_labels),pd.DataFrame(expected,index=row_labels,columns=col_labels)

band_totals=(group.groupby("size_band")["group_size"].sum().reindex(BAND_ORDER).astype(int))
source_totals=pd.Series({s:int(group[s].sum()) for s in sources})
BAND_TARGET,BAND_EXPECTED=rounded_transport_targets(role_caps.values,band_totals.values,ROLE_ORDER,BAND_ORDER)
SOURCE_TARGET,SOURCE_EXPECTED=rounded_transport_targets(role_caps.values,source_totals.values,ROLE_ORDER,sources)

# Each singleton has exactly one source; derive it from the 4-vector.
def one_source(row):
    active=[s for s in sources if int(row[s])==1]
    require(len(active)==1,"Singleton does not have exactly one source.")
    return active[0]

singleton=singleton.copy()
singleton["singleton_source"]=singleton.apply(one_source,axis=1)
singleton_supply=singleton.groupby("singleton_source").size().reindex(sources).astype(int)

# Fractional source×band targets and structural cell tolerances from 00E.
global_cells=pd.read_csv(AUDIT_GLOBAL_CELL)
global_cells["size_band"]=global_cells["size_band"].astype(str)
cell_global={(str(r["source_file"]),str(r["size_band"])):int(r["rows"])
             for _,r in global_cells.iterrows()}
cell_tol={(str(r["source_file"]),str(r["size_band"])):int(r["max_group_size_touching_cell"])
          for _,r in global_cells.iterrows() if int(r["rows"])>0}

role_fraction=role_caps/len(pure)
def cell_target(role,source,band):
    return float(role_fraction[role]*cell_global[(source,band)])

# Non-single total target is implied by singleton band target.
NS_TARGET=pd.Series({r:int(role_caps[r]-BAND_TARGET.loc[r,"1"]) for r in ROLE_ORDER})

atomic_csv(V3_ROOT/"target_role_x_band.csv",BAND_TARGET.reset_index().rename(columns={"index":"partition"}))
atomic_csv(V3_ROOT/"target_role_x_source.csv",SOURCE_TARGET.reset_index().rename(columns={"index":"partition"}))

print("Non-singleton row targets:")
display(NS_TARGET.rename("rows").reset_index())


Non-singleton row targets:


,index,rows
0,benign_weight_train,167841
1,benign_early_stop,33569
2,benign_threshold_calibration,50352
3,development,33419
4,final_test,50352


In [5]:

# ============================================================
# V3.4-4. COORDINATED MILP HELPERS
# ============================================================
TOP_BAND=">=100"
REMAINDER_BANDS=["50-99","20-49","10-19","5-9","3-4","2"]
R=len(ROLE_ORDER)
role_i={r:i for i,r in enumerate(ROLE_ORDER)}

def build_types(bands):
    sub=non_single.loc[non_single["size_band"].isin(bands)].copy()
    type_cols=["size_band","group_size"]+sources
    typ=(sub.groupby(type_cols,dropna=False,sort=True)
            .agg(type_count=("feature_fingerprint","size"))
            .reset_index())
    typ["type_id"]=np.arange(len(typ),dtype=int)
    sub=sub.merge(typ[type_cols+["type_id"]],on=type_cols,how="left",validate="many_to_one")
    require(sub["type_id"].notna().all(),f"Type mapping failed for bands={bands}")
    sub["type_id"]=sub["type_id"].astype(int)
    return typ,sub

def materialize_type_solution(typ,sub,X):
    local={}
    for t in range(len(typ)):
        g=sub.loc[sub["type_id"].eq(t)].sort_values(["tie_key","feature_fingerprint"])
        require(len(g)==int(typ.loc[t,"type_count"]),f"Type count mismatch t={t}")
        start=0
        for ri,r in enumerate(ROLE_ORDER):
            n=int(X[t,ri])
            take=g.iloc[start:start+n]
            for fp in take["feature_fingerprint"].astype(str):
                local[fp]=r
            start+=n
        require(start==len(g),f"Type materialization incomplete t={t}")
    return local

def solve_top_ge100(tie_salt="A", time_limit=240):
    bands=[TOP_BAND]
    typ,sub=build_types(bands)
    T=len(typ)
    NX=T*R

    # 5 roles × 4 sources, each with positive/negative L1 slack.
    cells=[(ri,r,s) for ri,r in enumerate(ROLE_ORDER) for s in sources]
    K=len(cells)
    NVAR=NX+2*K

    def xidx(t,ri): return t*R+ri
    def pidx(k): return NX+2*k
    def midx(k): return NX+2*k+1

    c=np.zeros(NVAR,dtype=float)
    lb=np.zeros(NVAR,dtype=float)
    ub=np.full(NVAR,np.inf,dtype=float)
    integrality=np.zeros(NVAR,dtype=int)

    # Integer type-count variables.
    for t in range(T):
        for ri,r in enumerate(ROLE_ORDER):
            j=xidx(t,ri)
            ub[j]=float(typ.loc[t,"type_count"])
            integrality[j]=1
            h=hashlib.sha256(
                f"{PARTITION_SEED}|{tie_salt}|{t}|{r}".encode()
            ).digest()
            c[j]=(int.from_bytes(h[:8],"big")/2**64)*1e-9

    # Normalize L1 source deviations by cell atomic scale.
    for k,(ri,r,s) in enumerate(cells):
        tol=max(1.0,float(cell_tol[(s,TOP_BAND)]))
        weight=(0.20 if (s,TOP_BAND) in EXEMPT_SOURCE_BAND else 1.0)/tol
        c[pidx(k)]=weight
        c[midx(k)]=weight

    rr=[];cc=[];dd=[];lo=[];hi=[];names=[];rowno=0
    def add(entries,L,U,name):
        nonlocal rowno
        for j,v in entries:
            if v!=0:
                rr.append(rowno);cc.append(int(j));dd.append(float(v))
        lo.append(float(L));hi.append(float(U));names.append(str(name));rowno+=1

    # Type conservation.
    for t in range(T):
        cnt=int(typ.loc[t,"type_count"])
        add([(xidx(t,ri),1.0) for ri in range(R)],cnt,cnt,f"type::{t}")

    # Exact >=100 role-row targets.
    for ri,r in enumerate(ROLE_ORDER):
        tgt=int(BAND_TARGET.loc[r,TOP_BAND])
        add([(xidx(t,ri),int(typ.loc[t,"group_size"])) for t in range(T)],
            tgt,tgt,f"role_band::{r}::{TOP_BAND}")

    # Frozen non-exempt source×band windows.
    for s in sources:
        tol=float(cell_tol[(s,TOP_BAND)])
        for ri,r in enumerate(ROLE_ORDER):
            if (s,TOP_BAND) in EXEMPT_SOURCE_BAND:
                continue
            tgt=cell_target(r,s,TOP_BAND)
            entries=[(xidx(t,ri),int(typ.loc[t,s])) for t in range(T)
                     if int(typ.loc[t,s])!=0]
            add(entries,max(0.0,tgt-tol),tgt+tol,
                f"source_band::{r}::{s}::{TOP_BAND}")

    # L1 equalities for source closeness, including exempt cells.
    for k,(ri,r,s) in enumerate(cells):
        tgt=cell_target(r,s,TOP_BAND)
        entries=[(xidx(t,ri),int(typ.loc[t,s])) for t in range(T)
                 if int(typ.loc[t,s])!=0]
        entries += [(pidx(k),-1.0),(midx(k),1.0)]
        add(entries,tgt,tgt,f"source_L1::{r}::{s}")

    # >400 row weight-training coverage.
    wt=role_i["benign_weight_train"]
    tids=typ.index[typ["group_size"].gt(400)].tolist()
    require(len(tids)>0,"No >400-row >=100 structural type.")
    add([(xidx(t,wt),1.0) for t in tids],1.0,np.inf,"weight_train_gt400")

    A=coo_matrix((dd,(rr,cc)),shape=(rowno,NVAR)).tocsr()
    cons=LinearConstraint(A,np.asarray(lo),np.asarray(hi))
    started=time.time()
    res=milp(
        c=c,integrality=integrality,bounds=Bounds(lb,ub),constraints=cons,
        options={"presolve":True,"time_limit":time_limit,"mip_rel_gap":0.01,"disp":True}
    )
    elapsed=time.time()-started
    require(res.x is not None,
            f">=100 solver returned no primal; status={res.status}; message={res.message}")

    xi=np.rint(res.x[:NX]).astype(int)
    require(np.max(np.abs(res.x[:NX]-xi))<1e-6,">=100 integer variables are nonintegral.")
    Ax=A@np.concatenate([xi,res.x[NX:]])
    require(np.all(Ax>=np.asarray(lo)-1e-6) and np.all(Ax<=np.asarray(hi)+1e-6),
            ">=100 independent hard-constraint verification failed.")

    X=xi.reshape(T,R)
    require(np.array_equal(X.sum(axis=1),typ["type_count"].to_numpy(dtype=int)),
            ">=100 type conservation failed.")

    info={
        "stage":"top_ge100",
        "tie_salt":tie_salt,
        "structural_types":int(T),
        "integer_variables":int(NX),
        "continuous_L1_variables":int(2*K),
        "constraints":int(rowno),
        "status":int(res.status),
        "success":bool(res.success),
        "message":str(res.message),
        "elapsed_seconds":float(elapsed),
        "objective":float(res.fun) if res.fun is not None else None,
        "independently_verified_feasible":True,
    }
    return typ,sub,X,info

def solve_remainder(top_source, top_rows, tie_salt="A", time_limit=900):
    typ,sub=build_types(REMAINDER_BANDS)
    T=len(typ); NX=T*R
    def vid(t,ri): return t*R+ri

    c=np.zeros(NX,dtype=float)
    for t in range(T):
        for ri,r in enumerate(ROLE_ORDER):
            h=hashlib.sha256(
                f"{PARTITION_SEED}|{tie_salt}|remainder|{t}|{r}".encode()
            ).digest()
            c[vid(t,ri)]=(int.from_bytes(h[:8],"big")/2**64)*1e-9

    lb=np.zeros(NX,dtype=float)
    ub=np.repeat(typ["type_count"].to_numpy(dtype=float),R)
    integrality=np.ones(NX,dtype=int)

    rr=[];cc=[];dd=[];lo=[];hi=[];names=[];rowno=0
    def add(entries,L,U,name):
        nonlocal rowno
        for j,v in entries:
            if v!=0:
                rr.append(rowno);cc.append(int(j));dd.append(float(v))
        lo.append(float(L));hi.append(float(U));names.append(str(name));rowno+=1

    # Every structural type fully assigned.
    for t in range(T):
        cnt=int(typ.loc[t,"type_count"])
        add([(vid(t,ri),1.0) for ri in range(R)],cnt,cnt,f"type::{t}")

    # IMPORTANT: use the actual frozen ±M_b tolerances, not v3.2's extra exactness.
    for b in REMAINDER_BANDS:
        tids=typ.index[typ["size_band"].astype(str).eq(b)].tolist()
        for ri,r in enumerate(ROLE_ORDER):
            tgt=int(BAND_TARGET.loc[r,b]); tol=int(M_B[b])
            add([(vid(t,ri),int(typ.loc[t,"group_size"])) for t in tids],
                tgt-tol,tgt+tol,f"role_band::{r}::{b}")

    # Frozen non-exempt source×multiplicity windows.
    for b in REMAINDER_BANDS:
        tids=typ.index[typ["size_band"].astype(str).eq(b)].tolist()
        for s in sources:
            if (s,b) in EXEMPT_SOURCE_BAND:
                continue
            if cell_global.get((s,b),0)==0:
                continue
            tol=float(cell_tol[(s,b)])
            for ri,r in enumerate(ROLE_ORDER):
                tgt=cell_target(r,s,b)
                entries=[(vid(t,ri),int(typ.loc[t,s])) for t in tids
                         if int(typ.loc[t,s])!=0]
                add(entries,max(0.0,tgt-tol),tgt+tol,
                    f"source_band::{r}::{s}::{b}")

    # Aggregate remainder rows must leave singleton band within its ±1 tolerance.
    expected_ns=pd.Series({
        r:int(role_caps[r]-BAND_TARGET.loc[r,"1"]) for r in ROLE_ORDER
    })
    for ri,r in enumerate(ROLE_ORDER):
        needed=int(expected_ns[r]-top_rows[r])
        entries=[(vid(t,ri),int(typ.loc[t,"group_size"])) for t in range(T)]
        add(entries,needed-1,needed+1,f"aggregate_remainder::{r}")

    # Residual source composition must leave each singleton source×band cell within ±1.
    for ri,r in enumerate(ROLE_ORDER):
        for s in sources:
            singleton_target=cell_target(r,s,"1")
            final_src=int(SOURCE_TARGET.loc[r,s])
            # residual singleton = final_src - top_source - remainder_source
            # enforce residual in [singleton_target-1, singleton_target+1]
            L=float(final_src-top_source.loc[r,s]-(singleton_target+1.0))
            U=float(final_src-top_source.loc[r,s]-(singleton_target-1.0))
            entries=[(vid(t,ri),int(typ.loc[t,s])) for t in range(T)
                     if int(typ.loc[t,s])!=0]
            add(entries,L,U,f"singleton_residual::{r}::{s}")

    A=coo_matrix((dd,(rr,cc)),shape=(rowno,NX)).tocsr()
    cons=LinearConstraint(A,np.asarray(lo),np.asarray(hi))

    def run(obj,seconds,gap):
        return milp(
            c=obj,integrality=integrality,bounds=Bounds(lb,ub),constraints=cons,
            options={"presolve":True,"time_limit":seconds,"mip_rel_gap":gap,"disp":True}
        )

    started=time.time()
    res=run(c,time_limit,1.0)
    used_zero=False
    if res.x is None:
        used_zero=True
        print("No primal with seeded tie objective; retrying remainder as pure feasibility.")
        res=run(np.zeros(NX,dtype=float),time_limit,1.0)
    elapsed=time.time()-started

    require(res.x is not None,
            f"Coordinated remainder returned no primal; status={res.status}; message={res.message}")
    x=np.rint(res.x).astype(int)
    require(np.max(np.abs(res.x-x))<1e-6,"Remainder primal is nonintegral.")
    Ax=A@x
    require(np.all(Ax>=np.asarray(lo)-1e-6) and np.all(Ax<=np.asarray(hi)+1e-6),
            "Remainder independent hard-constraint verification failed.")

    X=x.reshape(T,R)
    require(np.array_equal(X.sum(axis=1),typ["type_count"].to_numpy(dtype=int)),
            "Remainder type conservation failed.")

    info={
        "stage":"coordinated_remainder_2_to_99",
        "tie_salt":tie_salt,
        "structural_types":int(T),
        "integer_variables":int(NX),
        "constraints":int(rowno),
        "matrix_nonzeros":int(A.nnz),
        "status":int(res.status),
        "success":bool(res.success),
        "message":str(res.message),
        "elapsed_seconds":float(elapsed),
        "fallback_to_zero_objective":bool(used_zero),
        "objective":float(res.fun) if res.fun is not None else None,
        "independently_verified_feasible":True,
    }
    return typ,sub,X,info

print("v3.3 coordinated solver ready.")


v3.3 coordinated solver ready.


In [6]:

# ============================================================
# V3.4-5. SOLVE >=100 WITH SOURCE-BALANCED L1 OBJECTIVE
# ============================================================
assignment={}
solver_records=[]

print("\n=== SOLVING TOP BAND >=100 ===")
typ_top,sub_top,X_top,info_top=solve_top_ge100(tie_salt="A",time_limit=240)
local_top=materialize_type_solution(typ_top,sub_top,X_top)
assignment.update(local_top)

tmp=sub_top[["feature_fingerprint","group_size"]+sources].copy()
tmp["partition"]=tmp["feature_fingerprint"].astype(str).map(local_top)
top_source=(tmp.groupby("partition")[sources].sum()
              .reindex(ROLE_ORDER).fillna(0).astype(int))
top_rows=(tmp.groupby("partition")["group_size"].sum()
            .reindex(ROLE_ORDER).fillna(0).astype(int))

solver_records.append(info_top)
print(info_top)
print("\n>=100 source allocation:")
display(top_source)



=== SOLVING TOP BAND >=100 ===
{'stage': 'top_ge100', 'tie_salt': 'A', 'structural_types': 204, 'integer_variables': 1020, 'continuous_L1_variables': 40, 'constraints': 240, 'status': 1, 'success': False, 'message': 'Time limit reached. (HiGHS Status 13: Time limit reached)', 'elapsed_seconds': 240.0242862701416, 'objective': 0.3926739590740713, 'independently_verified_feasible': True}

>=100 source allocation:


,UNSW-NB15_1.csv,UNSW-NB15_2.csv,UNSW-NB15_3.csv,UNSW-NB15_4.csv
partition,,,,
benign_weight_train,234,2990,11836,10691
benign_early_stop,112,556,2364,2119
benign_threshold_calibration,111,881,3561,3172
development,0,644,2363,2120
final_test,114,860,3554,3197


In [7]:

# ============================================================
# V3.4-6. COORDINATED REMAINDER SOLVE (2–99) + SOLVER MANIFEST
# ============================================================
print("\n=== SOLVING COORDINATED REMAINDER",REMAINDER_BANDS,"===")
typ_rem,sub_rem,X_rem,info_rem=solve_remainder(
    top_source=top_source,
    top_rows=top_rows,
    tie_salt="A",
    time_limit=900,
)
local_rem=materialize_type_solution(typ_rem,sub_rem,X_rem)
assignment.update(local_rem)
solver_records.append(info_rem)

solver_manifest={
    "dataset":DATASET_NAME,
    "implementation_version":"v3.3_coordinated_feasibility",
    "scientific_policy_hash":FROZEN_POLICY["policy_hash"],
    "scipy_version":scipy.__version__,
    "subproblems":solver_records,
    "all_subproblems_independently_verified_feasible":True,
}
atomic_json(V3_ROOT/"coordinated_feasibility_solver_manifest.json",solver_manifest)

print(info_rem)
print("\nPASS: top and coordinated remainder are independently feasible.")



=== SOLVING COORDINATED REMAINDER ['50-99', '20-49', '10-19', '5-9', '3-4', '2'] ===
{'stage': 'coordinated_remainder_2_to_99', 'tie_salt': 'A', 'structural_types': 809, 'integer_variables': 4045, 'constraints': 984, 'matrix_nonzeros': 27745, 'status': 0, 'success': True, 'message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)', 'elapsed_seconds': 243.98796486854553, 'fallback_to_zero_objective': False, 'objective': 2.7888505240287583e-05, 'independently_verified_feasible': True}

PASS: top and coordinated remainder are independently feasible.


In [8]:

# ============================================================
# V3.4-7. SINGLETONS LAST — POLICY-CORRECT ±1 BAND TOLERANCE
# ============================================================
assigned_ns=non_single[["feature_fingerprint","group_size","size_band"]+sources].copy()
assigned_ns["new_partition"]=assigned_ns["feature_fingerprint"].astype(str).map(assignment)
require(assigned_ns["new_partition"].notna().all(),"Missing non-singleton assignment.")

# Actual solved non-single rows by role.
ns_role_rows=(assigned_ns.groupby("new_partition")["group_size"].sum()
              .reindex(ROLE_ORDER).fillna(0).astype(int))

# Actual singleton rows required to hit the exact role capacities.
actual_singleton_need=(role_caps-ns_role_rows).astype(int)
require((actual_singleton_need>=0).all(),"A role has negative singleton residual capacity.")

# Frozen singleton-band target and the only correct gate:
# |actual_singleton_need - target_singleton_rows| <= M_1 = 1.
target_singleton=BAND_TARGET.loc[ROLE_ORDER,"1"].astype(int)
singleton_band_deviation=(actual_singleton_need-target_singleton).astype(int)

require(
    (singleton_band_deviation.abs() <= int(M_B["1"])).all(),
    "Solved non-single totals leave a singleton-band residual outside the frozen ±1 tolerance: "
    f"{singleton_band_deviation.to_dict()}"
)

print("Singleton-band residual deviations (actual - target):")
display(singleton_band_deviation.rename("signed_deviation").reset_index())

# Exact overall source targets are still mandatory.
ns_source=(assigned_ns.groupby("new_partition")[sources].sum()
           .reindex(ROLE_ORDER).fillna(0).astype(int))
desired_singleton=(SOURCE_TARGET-ns_source).astype(int)

require((desired_singleton>=0).all().all(),"Negative singleton source residual.")

# The source residual counts must sum to the actual singleton row need in each role.
require(
    np.array_equal(
        desired_singleton.sum(axis=1).to_numpy(dtype=int),
        actual_singleton_need.to_numpy(dtype=int)
    ),
    "Singleton source residuals do not match actual role singleton capacities."
)

# Across roles, singleton source demand must exactly equal available singleton supply.
singleton_supply=singleton.groupby("singleton_source").size().reindex(sources).astype(int)
require(
    np.array_equal(
        desired_singleton.sum(axis=0).to_numpy(dtype=int),
        singleton_supply.to_numpy(dtype=int)
    ),
    "Singleton source residuals do not match global singleton source supply."
)

# Frozen source×singleton gate remains ±1 for every role/source.
singleton_source_gate_rows=[]
for r in ROLE_ORDER:
    for s in sources:
        actual=int(desired_singleton.loc[r,s])
        target=float(cell_target(r,s,"1"))
        deviation=actual-target
        passed=abs(deviation) <= 1.0 + 1e-9
        singleton_source_gate_rows.append({
            "partition":r,
            "source_file":s,
            "size_band":"1",
            "target_rows":target,
            "actual_rows":actual,
            "absolute_deviation":abs(deviation),
            "cell_tolerance":1.0,
            "pass":passed,
        })
        require(
            passed,
            f"Singleton source×band gate cannot be met for role={r}, source={s}: "
            f"actual={actual}, target={target}"
        )

SINGLETON_SOURCE_GATE=pd.DataFrame(singleton_source_gate_rows)
atomic_csv(V3_ROOT/"singleton_source_gate.csv",SINGLETON_SOURCE_GATE)

# Deterministic singleton assignment by source, then role.
for s in sources:
    ss=singleton.loc[singleton["singleton_source"].eq(s)].sort_values(
        ["tie_key","feature_fingerprint"]
    )
    start=0
    for r in ROLE_ORDER:
        n=int(desired_singleton.loc[r,s])
        take=ss.iloc[start:start+n]
        require(len(take)==n,f"Singleton shortage {s}/{r}")
        for fp in take["feature_fingerprint"].astype(str):
            assignment[fp]=r
        start+=n
    require(start==len(ss),f"Singleton source {s} not fully consumed.")

require(
    len(assignment)==len(group),
    f"Pure-benign assignment incomplete: {len(assignment)} vs {len(group)}"
)

group["new_partition"]=group["feature_fingerprint"].astype(str).map(assignment)
require(group["new_partition"].notna().all(),"Missing final pure-benign assignment.")

# Immediate exact role-capacity verification after singleton materialization.
post_caps=group.groupby("new_partition")["group_size"].sum().reindex(ROLE_ORDER).astype(int)
require(
    np.array_equal(post_caps.to_numpy(dtype=int), role_caps.to_numpy(dtype=int)),
    f"Exact role capacities not restored after singleton allocation: {(post_caps-role_caps).to_dict()}"
)

print("PASS: singleton allocation respects frozen ±1 band tolerance and restores exact role capacities.")


Singleton-band residual deviations (actual - target):


,partition,signed_deviation
0,benign_weight_train,0
1,benign_early_stop,1
2,benign_threshold_calibration,-1
3,development,1
4,final_test,-1


PASS: singleton allocation respects frozen ±1 band tolerance and restores exact role capacities.


In [9]:

# ============================================================
# V3.4-8. STRUCTURAL GATES BEFORE MEMBERSHIP WRITE
# ============================================================
# Exact role capacity
actual_caps=group.groupby("new_partition")["group_size"].sum().reindex(ROLE_ORDER).astype(int)
GATE_CAPACITY=np.array_equal(actual_caps.values,role_caps.values)

# Multiplicity all five roles
band_actual=(group.groupby(["new_partition","size_band"])["group_size"].sum()
                  .unstack(fill_value=0).reindex(index=ROLE_ORDER,columns=BAND_ORDER,fill_value=0).astype(int))
band_rows=[]
for r in ROLE_ORDER:
    for b in BAND_ORDER:
        target=int(BAND_TARGET.loc[r,b]); actual=int(band_actual.loc[r,b]); tol=int(M_B[b])
        band_rows.append({"partition":r,"size_band":b,"target_rows":target,"actual_rows":actual,
                          "signed_deviation":actual-target,"absolute_deviation":abs(actual-target),
                          "tolerance":tol,"pass":abs(actual-target)<=tol})
BAND_GATE=pd.DataFrame(band_rows)
GATE_MULTIPLICITY=bool(BAND_GATE["pass"].all())

# Full source vectors
gsrc=group[["feature_fingerprint","new_partition","size_band","group_size"]+sources].copy()
source_actual=gsrc.groupby("new_partition")[sources].sum().reindex(ROLE_ORDER).astype(int)
GATE_SOURCE_EXACT=np.array_equal(source_actual.values,SOURCE_TARGET.values)

# source×band
cell_records=[]
for r in ROLE_ORDER:
    rg=gsrc.loc[gsrc["new_partition"].eq(r)]
    for s in sources:
        for b in BAND_ORDER:
            actual=int(rg.loc[rg["size_band"].eq(b),s].sum())
            target=cell_target(r,s,b)
            exempt=(s,b) in EXEMPT_SOURCE_BAND
            tol=float(cell_tol.get((s,b),0 if cell_global.get((s,b),0)==0 else 1))
            passed=exempt or abs(actual-target)<=tol+1e-9
            cell_records.append({"partition":r,"source_file":s,"size_band":b,
                                 "target_rows":target,"actual_rows":actual,
                                 "absolute_deviation":abs(actual-target),
                                 "cell_tolerance":tol,"is_exempt":exempt,
                                 "pass_nonexempt":passed})
CELL_GATE=pd.DataFrame(cell_records)
GATE_SOURCE_BAND=bool(CELL_GATE["pass_nonexempt"].all())

# >=100 subcheck
large100=group.loc[group["size_band"].eq(">=100")]
LARGE_REPORT=(large100.groupby("new_partition")
              .agg(groups=("feature_fingerprint","size"),
                   rows=("group_size","sum"),
                   mean_group_size=("group_size","mean"),
                   max_group_size=("group_size","max"))
              .reindex(ROLE_ORDER).reset_index().rename(columns={"new_partition":"partition"}))
wt_gt400=int(((large100["new_partition"]=="benign_weight_train")&(large100["group_size"]>400)).sum())
GATE_WT_GT400=wt_gt400>=1

atomic_csv(V3_ROOT/"multiplicity_gate.csv",BAND_GATE)
atomic_csv(V3_ROOT/"source_gate_actual.csv",source_actual.reset_index().rename(columns={"new_partition":"partition"}))
atomic_csv(V3_ROOT/"source_x_multiplicity_gate.csv",CELL_GATE)
atomic_csv(V3_ROOT/"band_ge100_role_report.csv",LARGE_REPORT)

print("Exact role capacities:",GATE_CAPACITY)
print("Multiplicity all five roles:",GATE_MULTIPLICITY)
print("Exact overall pure source totals:",GATE_SOURCE_EXACT)
print("Source×multiplicity non-exempt cells:",GATE_SOURCE_BAND)
print("Weight-train >400 group:",GATE_WT_GT400)
display(BAND_GATE.loc[~BAND_GATE["pass"]])
display(CELL_GATE.loc[~CELL_GATE["pass_nonexempt"]])
display(LARGE_REPORT)

ALL_GATES=all([GATE_CAPACITY,GATE_MULTIPLICITY,GATE_SOURCE_EXACT,GATE_SOURCE_BAND,GATE_WT_GT400])
require(ALL_GATES,"One or more frozen v3 structural gates failed. Membership will NOT be written.")
print("\nALL PRE-MATERIALIZATION V3 STRUCTURAL GATES: PASS")


Exact role capacities: True
Multiplicity all five roles: True
Exact overall pure source totals: True
Source×multiplicity non-exempt cells: True
Weight-train >400 group: True


,partition,size_band,target_rows,actual_rows,signed_deviation,absolute_deviation,tolerance,pass


,partition,source_file,size_band,target_rows,actual_rows,absolute_deviation,cell_tolerance,is_exempt,pass_nonexempt


,partition,groups,rows,mean_group_size,max_group_size
0,benign_weight_train,118,25751,218.228814,713
1,benign_early_stop,34,5151,151.500000,252
2,benign_threshold_calibration,42,7725,183.928571,760
3,development,35,5127,146.485714,350
4,final_test,47,7725,164.361702,456



ALL PRE-MATERIALIZATION V3 STRUCTURAL GATES: PASS


In [10]:

# ============================================================
# V3.4-9. WRITE MEMBERSHIP; VERIFY NON-PURE PRESERVATION + ATOMICITY
# ============================================================
out_path=V3_ROOT/"partition_membership.parquet"
tmp_path=V3_ROOT/"partition_membership.parquet.tmp"
require(not out_path.exists(),"Refusing to overwrite existing v3 membership.")

pf=pq.ParquetFile(PARENT_MEMBERSHIP)
writer=None; rows_written=0
try:
    for batch in pf.iter_batches(batch_size=100_000):
        t=pa.Table.from_batches([batch])
        m=t.select(["feature_fingerprint","fingerprint_group_class","partition"]).to_pandas()
        pure_mask=m["fingerprint_group_class"].astype(str).eq(PURE_BENIGN_CLASS)
        newpart=m["partition"].astype(str).to_numpy(copy=True)
        if pure_mask.any():
            mapped=m.loc[pure_mask,"feature_fingerprint"].astype(str).map(assignment)
            require(mapped.notna().all(),"Unassigned pure fingerprint during materialization.")
            newpart[pure_mask.to_numpy()]=mapped.to_numpy(dtype=str)
        idx=t.schema.get_field_index("partition"); field=t.schema.field(idx)
        arr=pa.array(newpart)
        if not arr.type.equals(field.type): arr=arr.cast(field.type)
        t=t.set_column(idx,field,arr)
        if writer is None: writer=pq.ParquetWriter(tmp_path,t.schema,compression="zstd")
        writer.write_table(t); rows_written+=t.num_rows
finally:
    if writer is not None: writer.close()

require(rows_written==pf.metadata.num_rows,"Output row count mismatch.")
os.replace(tmp_path,out_path)
membership_sha=sha256_file(out_path)

check=pd.read_parquet(out_path,
    columns=["row_id","feature_fingerprint","fingerprint_group_class","partition","source_file"])
require(len(check)==len(struct),"Membership row count changed.")
require(check["row_id"].astype(str).equals(struct["row_id"].astype(str)),"Row identity/order changed.")

nonpure=~struct["fingerprint_group_class"].astype(str).eq(PURE_BENIGN_CLASS)
require(check.loc[nonpure,"partition"].astype(str).reset_index(drop=True).equals(
        struct.loc[nonpure,"partition"].astype(str).reset_index(drop=True)),
        "A non-pure row changed role.")
cross=int((check.groupby("feature_fingerprint")["partition"].nunique()>1).sum())
require(cross==0,f"Fingerprint crossings detected: {cross}")

print("PASS: v3 membership written and end-to-end verified.")
print("Membership SHA-256:",membership_sha)


PASS: v3 membership written and end-to-end verified.
Membership SHA-256: 273093382be4d417a7b45c837008b994e57644902dffa1194002d2f2d0f8d6de


In [11]:

# ============================================================
# V3.4-10. FREEZE INTEGRITY REPORT / PRE-02 HANDOFF
# ============================================================
report={
    "dataset":DATASET_NAME,
    "protocol_version":PROTOCOL_VERSION,
    "partition_version":MODEL_BRANCH_VERSION,
    "implementation_version":"v3.4_singleton_tolerance_fix",
    "status":"PASS",
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "scientific_policy_hash":FROZEN_POLICY["policy_hash"],
    "implementation_note_hash":implementation_note["implementation_note_hash"],
    "parent_membership_sha256":EXPECTED_PARENT_MEMBERSHIP_SHA,
    "partition_membership_sha256":membership_sha,
    "partition_seed":PARTITION_SEED,
    "pure_benign_rows":int(len(pure)),
    "pure_benign_fingerprints":int(len(group)),
    "role_pure_benign_capacities":{r:int(role_caps[r]) for r in ROLE_ORDER},
    "gates":{
        "exact_role_capacities":bool(GATE_CAPACITY),
        "multiplicity_all_five_roles":bool(GATE_MULTIPLICITY),
        "exact_overall_pure_source_totals":bool(GATE_SOURCE_EXACT),
        "source_x_multiplicity_nonexempt":bool(GATE_SOURCE_BAND),
        "weight_train_group_gt400":bool(GATE_WT_GT400),
        "nonpure_membership_preserved":True,
        "fingerprint_crossings":0,
    },
    "explicit_source_x_band_exemptions":[{"source_file":s,"size_band":b}
                                         for s,b in sorted(EXEMPT_SOURCE_BAND)],
    "weight_train_groups_gt400_in_band_ge100":int(wt_gt400),
    "solver_manifest":"coordinated_feasibility_solver_manifest.json",
    "formal_notebook01_authorized":False,
    "final_test_status":"FROZEN_AND_EVALUATION_SEALED",
    "next_authorized_stage":"PRE-02 v3 from scratch, followed by exactly one benign-only quantile-transfer smoke AE.",
    "stopping_rule":FROZEN_POLICY["stopping_rule"],
}
report["manifest_hash"]=canonical_hash({k:v for k,v in report.items() if k not in {"created_at_utc","manifest_hash"}})
atomic_json(V3_ROOT/"partition_integrity_report.json",report)

handoff={
    "dataset":DATASET_NAME,
    "partition_version":MODEL_BRANCH_VERSION,
    "implementation_version":"v3.4_singleton_tolerance_fix",
    "partition_membership_path":str(out_path),
    "partition_membership_sha256":membership_sha,
    "scientific_policy_hash":FROZEN_POLICY["policy_hash"],
    "fit_preprocessing_on":"benign_weight_train only",
    "transform_roles":ROLE_ORDER,
    "final_test_rule":"Mechanical transform only; no final-test outcome access before Notebook 07.",
    "formal_notebook01_authorized":False,
}
handoff["handoff_hash"]=canonical_hash(handoff)
atomic_json(V3_ROOT/"preprocessing_handoff_manifest.json",handoff)

print("\nPRE-01 v3.4 PARTITION INTEGRITY: PASS")
print("Final test: FROZEN AND EVALUATION-SEALED")
print("Formal Notebook 01: BLOCKED")
print("Next authorized stage: PRE-02 v3 from scratch")
print("\nReturn:")
for f in ["partition_integrity_report.json","multiplicity_gate.csv","source_gate_actual.csv",
          "source_x_multiplicity_gate.csv","singleton_source_gate.csv","band_ge100_role_report.csv",
          "milp_solver_manifest.json","preprocessing_handoff_manifest.json"]:
    print(" -",V3_ROOT/f)



PRE-01 v3.4 PARTITION INTEGRITY: PASS
Final test: FROZEN AND EVALUATION-SEALED
Formal Notebook 01: BLOCKED
Next authorized stage: PRE-02 v3 from scratch

Return:
 - /content/drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/partitioning_s8_s11_v3_source_multiplicity_seed20260914/partition_integrity_report.json
 - /content/drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/partitioning_s8_s11_v3_source_multiplicity_seed20260914/multiplicity_gate.csv
 - /content/drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/partitioning_s8_s11_v3_source_multiplicity_seed20260914/source_gate_actual.csv
 - /content/drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/partitioning_s8_s11_v3_source_multiplicity_seed20260914/source_x_multiplicity_gate.csv
 - /content/drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/partitioning_s8_s11_v3_source_multiplicity_seed20260914/singleton_source_gate.csv
 - /content/drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/partitioning_s8_s11_v3_source_multiplicity_seed20260914/band_ge